<a href="https://colab.research.google.com/github/JuanZapa7a/AINavalEngineering/blob/main/NB12_Training_Deep_Networks_Properly_ES.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **NB12 · Clase 12 — Entrenar redes profundas correctamente**

## Bloque 3: IA — Deep Learning (continuación)

`NB11` construyó una red neuronal que *funcionaba* — entrenaba, predecía, y quedaba cerca de los modelos clásicos de `NB08`. Esta clase convierte eso en una red de la que realmente te puedas *fiar*: una división correcta en entrenamiento/validación/test para monitorizar el entrenamiento, la lectura de las curvas de entrenamiento frente a validación para detectar el sobreajuste en el momento en que ocurre, dos técnicas de regularización (dropout, parada temprana) para combatirlo, y un vistazo a cómo la elección del optimizador cambia el propio entrenamiento. Mismo dataset real de Sonar que en `NB08`/`NB11`, para que todas las comparaciones sean directas.

### Objetivos de aprendizaje

Al terminar esta clase, el alumnado será capaz de:
- Explicar por qué las redes neuronales necesitan un conjunto de validación *durante* el entrenamiento, no solo un conjunto de test al final.
- Leer una curva de pérdida de entrenamiento frente a validación para diagnosticar el sobreajuste en el momento en que ocurre.
- Explicar y aplicar el dropout como técnica de regularización, y entender por qué solo actúa durante el entrenamiento.
- Implementar la parada temprana para detener el entrenamiento automáticamente en el punto adecuado.
- Comparar optimizadores (SGD, Adam, RMSprop) y explicar qué hace distinto cada uno.
- Combinar estas técnicas en un único modelo final, entrenado correctamente y evaluado con honestidad.

### Agenda (clase de 2 horas)

| # | Segmento de la clase | Duración aprox. | Tipo |
|---|---------------------|:---:|:---:|
| 1 | Repaso del Bloque 2-3 hasta ahora, hoja de ruta de hoy | 5 min | Teoría |
| 2 | Por qué las redes neuronales necesitan un conjunto de validación durante el entrenamiento | 10 min | Teoría |
| 3 | Práctica: división entrenamiento/validación, seguimiento de ambas curvas de pérdida | 15 min | Práctica |
| 4 | Diagnosticar el sobreajuste a partir de las curvas | 15 min | Teoría + Práctica |
| 5 | Dropout: teoría, estructura y una comparación práctica | 15 min | Teoría + Práctica |
| 6 | Parada temprana: teoría e implementación práctica | 15 min | Teoría + Práctica |
| 7 | Optimizadores: SGD, Adam y RMSprop comparados | 15 min | Teoría + Práctica |
| 8 | Poniéndolo todo junto: un modelo final entrenado correctamente | 20 min | Práctica |
| 9 | ¿Cuándo es la red neuronal la opción adecuada? | 5 min | Teoría |
| 10 | Resumen, tarea, próxima clase | 5 min | Teoría |

> Los tiempos son una orientación aproximada, no un guion estricto — no hay descansos programados. Si damos todo con tiempo de sobra, la clase termina antes; eso puede pasar y no hay problema.

---

## 1. Repaso: dónde estamos

- **Bloque 2** (`NB02`–`NB10`): el conjunto de herramientas clásico de ML, cerrado con un proyecto completo y ajustado.
- **`NB11`**: teoría de perceptrón → MLP, un primer clasificador real en PyTorch sobre el dataset Sonar, evaluado una vez frente a los modelos clásicos de `NB08`.
- **`NB12`** (hoy): el *proceso* de entrenamiento en sí — monitorización con validación, regularización, optimizadores — convirtiendo una red que simplemente funciona en una que generaliza.

---

## 2. Por qué las redes neuronales necesitan un conjunto de validación durante el entrenamiento

`NB11` solo monitorizaba la **pérdida de entrenamiento**. Eso nos dice que la red está ajustándose a *algo* — pero no si ese algo es el patrón real de los datos, o simplemente ruido en los 166 ejemplos de entrenamiento que vio. Este es exactamente el riesgo de sobreajuste de `NB07`/`NB08`, aplicado ahora a un modelo con miles de pesos entrenables en vez de un único parámetro `max_depth`.

Recuerda la división en tres partes de `NB07`:
- **Conjunto de entrenamiento**: a lo que se ajustan los pesos de la red.
- **Conjunto de validación**: monitorizado *durante todo* el entrenamiento, sin actualizar nunca los pesos a partir de él — nuestro sistema de alerta temprana frente al sobreajuste.
- **Conjunto de test**: se toca exactamente una vez, al final, igual que en `NB10`.

Para una red neuronal, "monitorizar el conjunto de validación" significa calcular su pérdida después de cada época (sin llamar a `loss.backward()` sobre ella) y `observar cómo las dos curvas — entrenamiento y validación — divergen o se mantienen juntas`.

---

## 3. Práctica: división entrenamiento/validación, seguimiento de ambas curvas de pérdida

Vuelve a cargar el mismo dataset real de Sonar que en `NB08`/`NB11`, pero esta vez divídelo en tres partes: 60% entrenamiento, 20% validación, 20% test.

In [ ]:
!wget -q -O sonar.csv https://raw.githubusercontent.com/JuanZapa7a/AINavalEngineering/main/Datasets/sonar.all-data

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

sonar = pd.read_csv("sonar.csv", header=None)
sonar.columns = [f"freq_{i}" for i in range(60)] + ["label"]

X = sonar.drop(columns="label").values
y = (sonar["label"] == "M").astype(int).values

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.25, random_state=42, stratify=y_train_full
)  # 0.25 of the remaining 80% = 20% of the total

print("Train:", X_train.shape, " Val:", X_val.shape, " Test:", X_test.shape)

Escala usando las estadísticas del **conjunto de entrenamiento únicamente** — ajustar el escalador con datos de validación o de test sería el mismo error de fuga que se señaló en `NB07`/`NB08`. Después convierte todo a tensores:

In [ ]:
import torch
import torch.nn as nn

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
X_val_t = torch.tensor(X_val_scaled, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.float32).view(-1, 1)
X_test_t = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

Reutiliza la arquitectura de `NB11`, y entrena durante muchas épocas — deliberadamente más de las que probablemente necesitemos — registrando *ambas* pérdidas en cada época:

In [ ]:
class SonarMLP(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(n_features, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
        )

    def forward(self, x):
        return self.layers(x)

torch.manual_seed(42)
model = SonarMLP(n_features=X_train_t.shape[1])
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

n_epochs = 400
train_losses, val_losses = [], []

for epoch in range(n_epochs):
    model.train()
    optimizer.zero_grad()
    outputs = model(X_train_t)
    loss = criterion(outputs, y_train_t)
    loss.backward()
    optimizer.step()
    train_losses.append(loss.item())

    model.eval()
    with torch.no_grad():
        val_loss = criterion(model(X_val_t), y_val_t)
    val_losses.append(val_loss.item())

Observa las llamadas a `model.train()` / `model.eval()` — un hábito que merece la pena adquirir ya, porque importa todavía más en cuanto el dropout entre en juego en la Parte 5. Dibuja ambas curvas juntas:

In [ ]:
import matplotlib.pyplot as plt

plt.plot(train_losses, label="Training loss")
plt.plot(val_losses, label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss (binary cross-entropy)")
plt.title("Training vs. validation loss, 400 epochs, no regularization")
plt.legend()
plt.show()

---

## 4. Diagnosticar el sobreajuste a partir de las curvas

**Lee tu propia gráfica** — es el equivalente, para redes neuronales, de la demostración de profundidad-frente-a-precisión del árbol de decisión de `NB08`:
- Si ambas curvas bajan juntas y se siguen aproximadamente la una a la otra, la red está generalizando — sigue así.
- Si la pérdida de entrenamiento sigue bajando mientras la de validación se estanca o empieza a *subir*, esa brecha **es** sobreajuste, visible exactamente igual que en el árbol de decisión — salvo que aquí el "mando" no es la profundidad, sino simplemente cuántas épocas dejamos entrenar, combinado con cuánta capacidad (capas × unidades) tiene la red respecto a cuántos datos (208 ejemplos en total) tiene para aprender.

Con una red pequeña pero de alta capacidad y solo 124 ejemplos de entrenamiento, `no sería sorprendente ver cierto grado de esta brecha hacia la época 400`. Las siguientes dos secciones cubren las dos soluciones más habituales.

**Pruébalo tú mismo**: convierte la lectura visual en dos números reales — la brecha final entre entrenamiento y validación, y la época exacta en la que la pérdida de validación alcanzó su mejor (más bajo) valor.

In [ ]:
import numpy as np

final_gap = val_losses[-1] - train_losses[-1]
best_val_epoch = int(np.argmin(val_losses))

print(f"Final train loss: {train_losses[-1]:.4f}, final val loss: {val_losses[-1]:.4f}, gap: {final_gap:.4f}")
print(f"Validation loss reached its minimum at epoch {best_val_epoch + 1} (of {n_epochs})")


---

## 5. Dropout

El **[dropout](https://en.wikipedia.org/wiki/Dropout_%28neural_networks%29)** desactiva aleatoriamente una fracción de las neuronas — distintas en cada pasada hacia delante — *solo durante el entrenamiento*. En cada pasada, `la red se ve obligada a hacer buenas predicciones sin depender de que una neurona concreta esté siempre presente`, lo que desincentiva que las neuronas se coadapten al ruido específico del conjunto de entrenamiento. En el momento de la evaluación, el dropout se desactiva y participan todas las neuronas (precisamente por eso importa `model.eval()` — es lo que le indica a las capas `Dropout` de PyTorch que dejen de desactivar neuronas).

Visualicemos cómo es un paso de entrenamiento con dropout, comparado con una pasada normal (totalmente conectada):

In [ ]:
import matplotlib.patches as patches
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(11, 5))

input_y = np.linspace(0, 4, 4)
hidden_y = np.linspace(0, 4, 5)
output_y = np.linspace(1, 3, 2)
dropped_hidden = {1, 3}

for ax, title, use_dropout in zip(
    axes, ["Without dropout", "With dropout (this training step)"], [False, True]
):
    for iy in input_y:
        for j, hy in enumerate(hidden_y):
            dropped = use_dropout and j in dropped_hidden
            ax.plot([0, 2], [iy, hy], color="lightgray" if dropped else "steelblue",
                     lw=0.7 if dropped else 1.2, zorder=1)

    for j, hy in enumerate(hidden_y):
        dropped = use_dropout and j in dropped_hidden
        for oy in output_y:
            ax.plot([2, 4], [hy, oy], color="lightgray" if dropped else "steelblue",
                     lw=0.7 if dropped else 1.2, zorder=1)

    for iy in input_y:
        ax.add_patch(patches.Circle((0, iy), 0.25, facecolor="lightblue", edgecolor="black", zorder=2))
    for j, hy in enumerate(hidden_y):
        dropped = use_dropout and j in dropped_hidden
        ax.add_patch(patches.Circle((2, hy), 0.25, facecolor="lightgray" if dropped else "lightcoral",
                                     edgecolor="black", zorder=2))
        if dropped:
            ax.text(2, hy, "x", ha="center", va="center", fontsize=12, color="dimgray", zorder=3)
    for oy in output_y:
        ax.add_patch(patches.Circle((4, oy), 0.25, facecolor="lightgreen", edgecolor="black", zorder=2))

    ax.set_xlim(-1, 5)
    ax.set_ylim(-1, 5)
    ax.axis("off")
    ax.set_title(title)

plt.tight_layout()
plt.show()

Cada pasada hacia delante durante el entrenamiento desactiva un subconjunto aleatorio *distinto* — el diagrama muestra un paso de ejemplo, no un patrón fijo. Ahora añade `nn.Dropout` a nuestra arquitectura y vuelve a entrenar, siguiendo ambas curvas exactamente como en la Parte 3:

In [ ]:
class SonarMLPDropout(nn.Module):
    def __init__(self, n_features, dropout_rate=0.3):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(n_features, 32),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(16, 1),
        )

    def forward(self, x):
        return self.layers(x)

torch.manual_seed(42)
dropout_model = SonarMLPDropout(n_features=X_train_t.shape[1], dropout_rate=0.3)
optimizer = torch.optim.Adam(dropout_model.parameters(), lr=0.001)

train_losses_do, val_losses_do = [], []
for epoch in range(n_epochs):
    dropout_model.train()
    optimizer.zero_grad()
    loss = criterion(dropout_model(X_train_t), y_train_t)
    loss.backward()
    optimizer.step()
    train_losses_do.append(loss.item())

    dropout_model.eval()
    with torch.no_grad():
        val_losses_do.append(criterion(dropout_model(X_val_t), y_val_t).item())

plt.plot(train_losses_do, label="Training loss (with dropout)")
plt.plot(val_losses_do, label="Validation loss (with dropout)")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs. validation loss, with dropout")
plt.legend()
plt.show()

**Compara esta gráfica con la de la Parte 4**: ¿se abre la brecha entre entrenamiento y validación más despacio, o se mantiene menor en general? El dropout suele hacer que la pérdida de entrenamiento baje *más despacio* (la red no puede ajustarse a los datos de entrenamiento con tanta avidez) a cambio de una pérdida de validación que se sostiene mejor — un intercambio directo y visible de un poco de rendimiento en entrenamiento por mejor generalización.

> **Para saber más**: [Dropout (Wikipedia)](https://en.wikipedia.org/wiki/Dropout_%28neural_networks%29) · [Regularización (Wikipedia)](https://en.wikipedia.org/wiki/Regularization_%28mathematics%29) · [documentación de `torch.nn.Dropout`](https://pytorch.org/docs/stable/generated/torch.nn.Dropout.html).

---

## 6. Parada temprana

El dropout cambia *qué* puede ajustar la red; la **[parada temprana](https://en.wikipedia.org/wiki/Early_stopping)** cambia *cuánto tiempo* dejamos que lo intente. La idea: guardar una copia de los pesos del modelo cada vez que la pérdida de validación alcanza un nuevo mínimo, y si no mejora durante un número determinado de épocas (la **paciencia**), detener el entrenamiento y restaurar esa mejor copia — `en vez de entrenar a ciegas durante un número fijo de épocas sin importar qué esté haciendo la curva de validación`.

In [ ]:
import copy

torch.manual_seed(42)
es_model = SonarMLPDropout(n_features=X_train_t.shape[1], dropout_rate=0.3)
optimizer = torch.optim.Adam(es_model.parameters(), lr=0.001)

patience = 25
best_val_loss = float("inf")
patience_counter = 0
best_state = None
max_epochs = 400

for epoch in range(max_epochs):
    es_model.train()
    optimizer.zero_grad()
    loss = criterion(es_model(X_train_t), y_train_t)
    loss.backward()
    optimizer.step()

    es_model.eval()
    with torch.no_grad():
        val_loss = criterion(es_model(X_val_t), y_val_t).item()

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(es_model.state_dict())
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch + 1} (best validation loss: {best_val_loss:.4f})")
            break

es_model.load_state_dict(best_state)

**Pruébalo tú mismo**: compara la época en la que se detuvo la parada temprana con el punto de divergencia de la Sección 4 (`best_val_epoch`, calculado antes) — usan modelos distintos (este tiene dropout) así que no coincidirán exactamente, pero deberían estar en un rango parecido si ambos responden a la misma señal de sobreajuste subyacente.

In [ ]:
print(f"Early stopping halted at epoch {epoch + 1}; the best validation loss was seen roughly "
      f"{patience} epochs earlier, around epoch {epoch + 1 - patience}.")
print(f"Section 4's val-loss-minimum epoch (no dropout): {best_val_epoch + 1}")


**Lee tu propia salida**: ¿se detuvo el entrenamiento antes de llegar a `max_epochs`? Si es así, ¿aproximadamente en qué época? — ¿coincide con el punto donde las curvas originales de la Parte 3 empezaron a divergir? `best_state` guarda ahora los pesos de la *mejor* época de validación, no necesariamente la última — exactamente el modelo que en realidad queremos conservar.

> **Para saber más**: [Parada temprana (Wikipedia)](https://en.wikipedia.org/wiki/Early_stopping).

---

## 7. Optimizadores: SGD, Adam y RMSprop comparados

`NB11` usó **Adam** sin dar muchas explicaciones. Los tres optimizadores habituales implementan la misma idea central de la Parte 5 de `NB11` (mover los pesos en contra del gradiente), pero difieren en *cómo*:

| Optimizador | Idea central | Comportamiento típico |
|---|---|---|
| **[SGD](https://en.wikipedia.org/wiki/Stochastic_gradient_descent)** (con momentum) | Pasos de gradiente simples, acumulando opcionalmente una "velocidad" a partir de pasos anteriores | Simple, bien entendido, a menudo necesita un ajuste más cuidadoso de la tasa de aprendizaje |
| **Adam** | Adapta la tasa de aprendizaje *por peso*, usando estimaciones móviles de los gradientes recientes | Convergencia rápida, buena opción por defecto para muchos problemas, usado en `NB11`/`NB12` hasta ahora |
| **RMSprop** | También adapta la tasa de aprendizaje por peso, usando una media móvil de los gradientes al cuadrado | Un predecesor/pariente de Adam, a menudo competitivo en problemas de tipo recurrente |

Comparemos directamente sus curvas de pérdida de entrenamiento, con la misma arquitectura y los mismos datos, manteniendo todo lo demás fijo:

In [ ]:
def train_and_track(optimizer_name, n_epochs=150):
    torch.manual_seed(42)
    m = SonarMLP(n_features=X_train_t.shape[1])
    if optimizer_name == "SGD":
        opt = torch.optim.SGD(m.parameters(), lr=0.01, momentum=0.9)
    elif optimizer_name == "Adam":
        opt = torch.optim.Adam(m.parameters(), lr=0.001)
    else:
        opt = torch.optim.RMSprop(m.parameters(), lr=0.001)

    losses = []
    for _ in range(n_epochs):
        m.train()
        opt.zero_grad()
        loss = criterion(m(X_train_t), y_train_t)
        loss.backward()
        opt.step()
        losses.append(loss.item())
    return losses

optimizer_curves = {name: train_and_track(name) for name in ["SGD", "Adam", "RMSprop"]}

for name, losses in optimizer_curves.items():
    plt.plot(losses, label=name)
plt.xlabel("Epoch")
plt.ylabel("Training loss")
plt.title("Optimizer comparison (same architecture, same data)")
plt.legend()
plt.show()

**Pruébalo tú mismo**: la gráfica solo muestra la pérdida de *entrenamiento* — amplía la comparación también a la pérdida final de *validación* de cada optimizador, para poder comparar directamente "convergencia de entrenamiento más rápida" y "mejor generalización", en vez de asumir que son lo mismo.

In [ ]:
def train_and_track_with_val(optimizer_name, n_epochs=150):
    torch.manual_seed(42)
    m = SonarMLP(n_features=X_train_t.shape[1])
    if optimizer_name == "SGD":
        opt = torch.optim.SGD(m.parameters(), lr=0.01, momentum=0.9)
    elif optimizer_name == "Adam":
        opt = torch.optim.Adam(m.parameters(), lr=0.001)
    else:
        opt = torch.optim.RMSprop(m.parameters(), lr=0.001)

    for _ in range(n_epochs):
        m.train()
        opt.zero_grad()
        loss = criterion(m(X_train_t), y_train_t)
        loss.backward()
        opt.step()

    m.eval()
    with torch.no_grad():
        final_val_loss = criterion(m(X_val_t), y_val_t).item()
    return final_val_loss

for name in ["SGD", "Adam", "RMSprop"]:
    print(f"{name}: final validation loss = {train_and_track_with_val(name):.4f}")


**Interpreta tu propia gráfica**: ¿qué optimizador alcanza una pérdida baja más rápido? ¿Se queda SGD por detrás de los dos métodos adaptativos, como predeciría la tabla anterior? Prueba a cambiar la tasa de aprendizaje de SGD (`lr=0.01` → `lr=0.1` o `lr=0.001`) y vuelve a ejecutar — ¿cuán sensible es el SGD simple a esa elección, comparado con Adam/RMSprop?

> **Para saber más**: [Descenso de gradiente estocástico (Wikipedia)](https://en.wikipedia.org/wiki/Stochastic_gradient_descent) · [documentación de `torch.optim.SGD`](https://pytorch.org/docs/stable/generated/torch.optim.SGD.html) · [documentación de `torch.optim.RMSprop`](https://pytorch.org/docs/stable/generated/torch.optim.RMSprop.html).

---

## 8. Uniéndolo todo: un modelo final entrenado correctamente

Combina todo: dropout, parada temprana, Adam — y solo entonces, evalúa sobre el conjunto de test que no hemos tocado desde la Parte 3.

In [ ]:
torch.manual_seed(42)
final_model = SonarMLPDropout(n_features=X_train_t.shape[1], dropout_rate=0.3)
optimizer = torch.optim.Adam(final_model.parameters(), lr=0.001)

best_val_loss = float("inf")
patience_counter = 0
best_state = None

for epoch in range(max_epochs):
    final_model.train()
    optimizer.zero_grad()
    loss = criterion(final_model(X_train_t), y_train_t)
    loss.backward()
    optimizer.step()

    final_model.eval()
    with torch.no_grad():
        val_loss = criterion(final_model(X_val_t), y_val_t).item()

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(final_model.state_dict())
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Stopped at epoch {epoch + 1}")
            break

final_model.load_state_dict(best_state)

Evalúa el modelo final sobre el conjunto de test que no hemos tocado desde la Parte 3:

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

final_model.eval()
with torch.no_grad():
    test_probs = torch.sigmoid(final_model(X_test_t))
    test_preds = (test_probs > 0.5).float()

y_pred_np = test_preds.numpy().ravel()
y_test_np = y_test_t.numpy().ravel()

print(confusion_matrix(y_test_np, y_pred_np))
print()
print(classification_report(y_test_np, y_pred_np, target_names=["Rock", "Mine"]))

**Pruébalo tú mismo**: haz concreta la comparación de la reflexión de más abajo. Evalúa el modelo original sin regularizar de la Sección 3 (nunca antes tocado por el conjunto de test) sobre este mismo conjunto de test y compara su precisión directamente con la de `final_model`.

In [ ]:
model.eval()
with torch.no_grad():
    unreg_test_preds = (torch.sigmoid(model(X_test_t)) > 0.5).float()
unreg_test_accuracy = (unreg_test_preds == y_test_t).float().mean().item()
final_test_accuracy = (test_preds == y_test_t).float().mean().item()

print(f"Section 3 model (no regularization) test accuracy: {unreg_test_accuracy:.3f}")
print(f"Section 8 model (dropout + early stopping)  test accuracy: {final_test_accuracy:.3f}")


Mira cuál gana en tu ejecución. Con un conjunto de test de solo ~40 filas, una diferencia de una o dos predicciones puede cambiar qué modelo parece mejor — si la regularización no gana aquí claramente, es un resultado real y honesto, no un fallo del método; el argumento de la Sección 9 sobre datasets tabulares pequeños se aplica tanto dentro de este notebook como frente a los modelos clásicos de `NB08`.

**La comparación de tres que realmente importa**: ¿cómo se compara esta precisión de test con (a) el MLP sin ajustar de `NB11`, y (b) el mejor modelo clásico de `NB08`? Una red correctamente regularizada *debería* sostenerse al menos igual de bien que la versión de `NB11`, e idealmente cerrar parte de la brecha con los modelos clásicos ajustados de `NB08` — aunque en un dataset tan pequeño (208 filas en total), `los métodos clásicos basados en árboles suelen seguir siendo muy competitivos frente a las redes neuronales, que tienden a necesitar más datos para mostrar una ventaja clara`.

---

## 9. ¿Cuándo es la red neuronal la opción adecuada?

Después de dos clases de comparación real frente a los modelos clásicos de `NB08` sobre el *mismo* dataset tabular, una conclusión justa: para problemas pequeños, tabulares y con las características ya diseñadas a mano como el nuestro, `los ensembles basados en árboles suelen ser igual de buenos, más rápidos de entrenar y mucho más fáciles de ajustar`. Las redes neuronales justifican su complejidad adicional principalmente cuando:

| Situación | Por qué ayudan las redes neuronales |
|---|---|
| Datos en crudo, no estructurados (imágenes, audio, secuencias largas) | El aprendizaje automático de características sustituye al diseño manual — la motivación de la Parte 2 de `NB11` |
| Datasets muy grandes | Las redes profundas en general siguen mejorando con más datos; los modelos clásicos alcanzan antes su techo |
| Hay transfer learning disponible | Una red preentrenada en un dataset enorme puede adaptarse a uno pequeño — la clase de CNN de `NB13` usará exactamente esto |

Esto prepara exactamente el terreno para `NB13`: nuestro próximo dataset real (imágenes submarinas) es exactamente el caso de "datos en crudo, no estructurados" en el que el aprendizaje automático de características de una CNN supera genuinamente a las características tabulares diseñadas a mano.

---

## Resumen de la clase

- Las redes neuronales necesitan un conjunto de validación monitorizado *durante* el entrenamiento, no solo un conjunto de test final — se sigue en cada época, no solo una vez.
- Una pérdida de entrenamiento que sigue bajando mientras la de validación se estanca o sube es sobreajuste, visible directamente en las dos curvas.
- El dropout desactiva neuronas aleatoriamente solo durante el entrenamiento, desincentivando la dependencia excesiva de una neurona concreta; `model.train()`/`model.eval()` controlan si está activo.
- La parada temprana conserva los pesos de la mejor pérdida de validación y detiene el entrenamiento automáticamente, en vez de adivinar un número fijo de épocas.
- SGD, Adam y RMSprop implementan el descenso de gradiente de formas distintas — Adam/RMSprop adaptan su tasa de aprendizaje por peso y normalmente convergen más rápido.
- En datasets tabulares pequeños, una red neuronal correctamente regularizada es competitiva con un modelo clásico ajustado — pero no automáticamente mejor; la ventaja real aparece con datos en crudo, no estructurados, o muchos más datos.

## Para la próxima clase (NB13)

Pasamos a las **redes neuronales convolucionales (CNN)**: la arquitectura diseñada específicamente para datos de imagen, aplicada a imágenes reales de inspección submarina — exactamente el tipo de datos en crudo y no estructurados donde el argumento de la Parte 9 a favor del deep learning realmente compensa.

## Tarea / Ideas de práctica

1. Cambia `dropout_rate` a `0.1` y a `0.5` en la Parte 5 — ¿cómo cambia cada valor la brecha entre la pérdida de entrenamiento y la de validación comparado con `0.3`?
2. Cambia `patience` en la Parte 6 a `5` y a `50` — ¿cómo cambia la época en la que se detiene el entrenamiento, y cómo cambia la precisión final de test?
3. Añade un cuarto optimizador a la comparación de la Parte 7, `torch.optim.Adagrad` — ¿cómo se compara su curva con las otras tres?
4. Combina el dropout con una red *más grande* (por ejemplo, capas ocultas de 128 y 64 en vez de 32 y 16) — ¿permite el dropout que la red más grande evite el sobreajuste tan eficazmente como lo hizo la más pequeña?
5. Explica con tus propias palabras por qué calculamos la pérdida de validación dentro de un bloque `with torch.no_grad():` — ¿qué saldría mal (o simplemente se desperdiciaría esfuerzo) si no lo hiciéramos?

> ***Como siempre: una pérdida de entrenamiento más baja no es un logro en sí misma — solo importa si el rendimiento en validación y test mejora junto con ella.***